# L03 – Träningsloopen, loss och optimerare

**Tid:** ca 2 h · **Del A: Grunder**

## Mål
1. Skriva en träningsloop och förklara varje rad i den: **forward, loss, backward, step**.
2. Välja **loss-funktion** efter problemtyp (MSE, CrossEntropy, BCEWithLogits) och förstå vad **logits** är.
3. Förstå skillnaden mellan **SGD** och **Adam**.
4. Se hur **learning rate** och **batchstorlek** påverkar träningen.

**Var i arbetsflödet?** Steg 6, *träna*. I sklearn gömmer sig allt det här i `.fit()`. I PyTorch skriver du loopen själv, och det är just därför du kan styra och felsöka den.

In [ ]:
import random
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

def set_seed(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Data: California Housing

Målet är att förutsäga medianpriset på bostäder i ett distrikt (i enheten 100 000 dollar) från 8 features. Det är ett **regressionsproblem**. Vi följer arbetsflödet från L01: dela upp och standardisera med statistik **bara från train**.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = fetch_california_housing(return_X_y=True)
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42)

scaler = StandardScaler().fit(X_train)          # bara train!
X_train_s, X_val_s, X_test_s = (scaler.transform(a) for a in (X_train, X_val, X_test))

def to_tensors(X, y):
    # float32 för GPU-hastighet; y får shape (N, 1) för att matcha modellens utdata
    return (torch.tensor(X, dtype=torch.float32, device=device),
            torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1))

Xtr, ytr = to_tensors(X_train_s, y_train)
Xva, yva = to_tensors(X_val_s, y_val)
print("train:", Xtr.shape, ytr.shape, " val:", Xva.shape)

### Baslinjer
Innan vi tränar något i PyTorch behöver vi veta vad som är bra. Vi använder två baslinjer: att alltid gissa medelvärdet och linjär regression i sklearn. Vi mäter med **RMSE** (root mean squared error), som har samma enhet som målet.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

rmse_mean = root_mean_squared_error(y_val, np.full_like(y_val, y_train.mean()))
lin = LinearRegression().fit(X_train_s, y_train)
rmse_lin = root_mean_squared_error(y_val, lin.predict(X_val_s))
print(f"Baslinje 'gissa medel'  val RMSE: {rmse_mean:.4f}")
print(f"sklearn LinearRegression val RMSE: {rmse_lin:.4f}")

## 2. Linjär regression för hand

En linjär modell är $\hat{y} = Xw + b$. Vi tränar den med gradient descent från L02. **Loss** är MSE (mean squared error):

$$\text{MSE} = \frac{1}{N}\sum_i (\hat{y}_i - y_i)^2$$

**Varför MSE för regression?** Den straffar stora fel extra hårt (kvadraten), och den är deriverbar överallt. Det är samma mål som `LinearRegression` minimerar.

In [ ]:
set_seed(42)
w = torch.zeros(8, 1, device=device, requires_grad=True)
b = torch.zeros(1, device=device, requires_grad=True)
lr = 0.1

for epoch in range(200):
    pred = Xtr @ w + b                      # 1. forward: gör prediktioner
    loss = ((pred - ytr) ** 2).mean()       # 2. loss: hur fel är vi?
    loss.backward()                         # 3. backward: gradienter för w och b
    with torch.no_grad():                   # 4. step: uppdatera parametrarna
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()          # 5. nollställ gradienterna
    if epoch % 40 == 0:
        print(f"epok {epoch:3d}: train MSE {loss.item():.4f}")

with torch.no_grad():
    rmse_manual = ((Xva @ w + b - yva) ** 2).mean().sqrt().item()
print(f"Vår modell val RMSE: {rmse_manual:.4f}   (sklearn: {rmse_lin:.4f})")

Vi kommer nära sklearn. Skillnaden är att sklearn löser problemet exakt (minsta kvadrat-metoden), medan vi tar många små steg mot lösningen.

En **epok** betyder att modellen har sett hela träningsdatan en gång. Här gör vi ett steg per epok, eftersom vi använder all data på en gång (*full batch*).

## 3. Samma sak med PyTorch-byggstenar

I praktiken skriver vi inte $Xw + b$ själva. PyTorch har tre byggstenar:

| Byggsten | Vad | Ersätter |
|---|---|---|
| `nn.Linear(8, 1)` | ett lager med vikter `w` och `b` | `Xtr @ w + b` |
| `nn.MSELoss()` | loss-funktionen | `((pred - y)**2).mean()` |
| `torch.optim.SGD(...)` | uppdaterar alla parametrar | `w -= lr * w.grad` och `zero_()` |

Träningsloopen nedan är **standardmönstret i PyTorch** och används i hela kursen:

In [ ]:
set_seed(42)
model = nn.Linear(8, 1).to(device)          # modellen ska till samma device som datan
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(200):
    optimizer.zero_grad()           # nollställ gamla gradienter
    pred = model(Xtr)               # forward
    loss = loss_fn(pred, ytr)       # loss
    loss.backward()                 # backward
    optimizer.step()                # uppdatera vikterna

with torch.no_grad():
    rmse_nn = loss_fn(model(Xva), yva).sqrt().item()
print(f"nn.Linear + SGD val RMSE: {rmse_nn:.4f}")
print("Inlärda parametrar:", [p.shape for p in model.parameters()])

## 4. Vad händer om … learning rate ändras?

**Gissa först:** vilken av `lr = 0.001, 0.01, 0.1, 1.0` ger lägst loss efter 100 epoker? Händer något konstigt med någon av dem?

In [ ]:
def train_full_batch(lr, epochs=100, opt_class=torch.optim.SGD):
    set_seed(42)
    model = nn.Linear(8, 1).to(device)
    optimizer = opt_class(model.parameters(), lr=lr)
    losses = []
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(Xtr), ytr)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

plt.figure(figsize=(8, 4))
for lr in [0.001, 0.01, 0.1, 1.0]:
    losses = train_full_batch(lr)
    print(f"lr={lr:<6} slut-loss: {losses[-1]:.4g}")
    plt.plot(losses, label=f"lr={lr}")
plt.yscale("log"); plt.xlabel("epok"); plt.ylabel("train MSE (log-skala)")
plt.legend(); plt.title("Learning rate styr steglängden"); plt.show()

- **För låg** (0.001): loss sjunker, men alldeles för långsamt.
- **Lagom** (0.1): snabbt ned till en låg nivå.
- **För hög** (1.0): stegen skjuter över minimum, och loss ökar för varje steg i stället för att minska (divergens) tills den blir `inf` och sedan `nan`. Kurvan försvinner då ur diagrammet.

**Tumregel:** om loss ökar eller blir `nan` i början av träningen, sänk learning rate med en faktor 10.

## 5. Mini-batcher och batchstorlek

Hittills har vi använt all data i varje steg. Med stora dataset (som bilder) går det inte: datan får inte plats i GPU-minnet, och varje steg blir långsamt. I stället delar vi upp datan i **mini-batcher**:

- **En batch** är en delmängd av träningsdatan, till exempel 64 exempel.
- **Ett steg** är en uppdatering av vikterna efter en batch.
- **En epok** är så många steg som behövs för att gå igenom all data en gång.

Gradienten från en batch är en *brusig uppskattning* av den fulla gradienten. Det är "stochastic" i *stochastic gradient descent*. I L04 sköter `DataLoader` batchningen. Här gör vi den för hand för att se vad som händer:

In [ ]:
def train_minibatch(batch_size, epochs=10, lr=0.001):
    set_seed(42)
    model = nn.Linear(8, 1).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    n = len(Xtr)
    t0 = time.time()
    for _ in range(epochs):
        perm = torch.randperm(n, device=device)         # blanda varje epok
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(Xtr[idx]), ytr[idx])
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        val_rmse = loss_fn(model(Xva), yva).sqrt().item()
    steps = epochs * ((n + batch_size - 1) // batch_size)
    return val_rmse, steps, time.time() - t0

print(f"{'batch':>6} {'steg':>7} {'val RMSE':>9} {'tid (s)':>8}")
for bs in [8, 64, 512, len(Xtr)]:
    rmse, steps, sec = train_minibatch(bs)
    print(f"{bs:>6} {steps:>7} {rmse:>9.4f} {sec:>8.2f}")

Med **samma antal epoker**:
- **Liten batch** ger många steg per epok, så modellen hinner lära sig mer. Stegen är brusiga och träningen tar längre tid, eftersom GPU:n inte utnyttjas fullt.
- **Stor batch** ger få steg. Varje steg är exakt men dyrt, och modellen hinner inte lika långt. Stora batcher behöver ofta högre learning rate.

**Tumregel:** börja med **32–256**. Batchstorlek och learning rate hänger ihop: ändrar du den ena kan du behöva justera den andra. Bruset från små batcher kan dessutom hjälpa modellen att generalisera lite bättre.

**Vad händer om … learning rate höjs till 0,003?** Testa `train_minibatch(8, lr=0.003)`. Den här datan har några extrema värden: vissa features ligger upp till 80 standardavvikelser från medel efter standardiseringen. En liten batch som råkar innehålla ett sådant exempel ger ett enormt steg, och träningen kan explodera. Små batcher och hög learning rate är en riskabel kombination.

In [ ]:
print("batch 8,  lr=0.003:", train_minibatch(8, lr=0.003)[0])
print("batch 64, lr=0.003:", train_minibatch(64, lr=0.003)[0])

## 6. Optimerare: SGD kontra Adam

- **SGD** tar samma steglängd (`lr`) för alla parametrar. Den är enkel och fungerar bra, men den är känslig för valet av learning rate.
- **Adam** anpassar steglängden *per parameter* utifrån tidigare gradienter. Parametrar med små, stabila gradienter får relativt större steg. Den fungerar bra direkt med standardvärdet `lr=1e-3` i de flesta problem.

**Varför Adam som standardval?** Den kräver minst justering, och därför börjar de flesta projekt med den. SGD med momentum används ofta när man vill pressa ut sista procenten på bildmodeller, men den kräver mer justering av learning rate.

In [ ]:
plt.figure(figsize=(8, 4))
for name, opt, lr in [("SGD lr=0.01", torch.optim.SGD, 0.01),
                      ("SGD lr=0.1", torch.optim.SGD, 0.1),
                      ("Adam lr=0.01", torch.optim.Adam, 0.01),
                      ("Adam lr=0.1", torch.optim.Adam, 0.1)]:
    plt.plot(train_full_batch(lr, epochs=200, opt_class=opt), label=name)
plt.yscale("log"); plt.xlabel("epok"); plt.ylabel("train MSE"); plt.legend(); plt.show()

I det här enkla, väl skalade problemet är SGD med bra learning rate fullt jämförbar med Adam. Fördelen med Adam syns mest i djupa nätverk med många olika sorters parametrar.

## 7. Loss för klassificering: logits och CrossEntropy

Vid klassificering med $K$ klasser ger modellen $K$ tal per exempel. De kallas **logits**: obegränsade tal, där högre betyder "mer troligt". **Softmax** gör om dem till sannolikheter som summerar till 1:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

**Cross-entropy** är minus logaritmen av sannolikheten för den *rätta* klassen: $-\log p_{\text{rätt}}$.
- Om modellen ger rätt klass hög sannolikhet blir loss nära 0.
- Om modellen är säker men har fel blir loss mycket stor.

Det är samma loss som `LogisticRegression` minimerar (*log loss*).

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0],    # exempel 1: modellen tror på klass 0
                       [0.1, 0.2, 3.0]])   # exempel 2: modellen tror på klass 2
targets = torch.tensor([0, 1])             # rätt svar: klass 0 respektive klass 1

probs = torch.softmax(logits, dim=1)
print("sannolikheter:\n", probs.round(decimals=3))
manual = -torch.log(probs[torch.arange(2), targets]).mean()
print("CE för hand:", manual.item())
print("nn.CrossEntropyLoss:", nn.CrossEntropyLoss()(logits, targets).item())

**Viktigt:** `nn.CrossEntropyLoss` tar **logits**, inte sannolikheter. Den gör softmax själv, på ett numeriskt stabilt sätt. Därför har ett klassificeringsnätverk **ingen softmax i sista lagret** under träning. Etiketterna ska vara `int64` med klassindex, inte one-hot.

**Vilken loss när?**

| Problem | Sista lagret | Loss | sklearn-motsvarighet |
|---|---|---|---|
| Regression | `Linear(…, 1)` | `nn.MSELoss` (eller `nn.L1Loss` om du har outliers) | `LinearRegression` |
| Binär klassificering | `Linear(…, 1)` (en logit) | `nn.BCEWithLogitsLoss` | `LogisticRegression` |
| Flera klasser | `Linear(…, K)` | `nn.CrossEntropyLoss` | `LogisticRegression(multi_class)` |

## 8. Övningar

### Övning 1 – CrossEntropy för hand
Beräkna cross-entropy för hand (softmax, logaritm och medelvärde) och kontrollera mot `nn.CrossEntropyLoss`.

In [ ]:
logits = torch.tensor([[1.0, 2.0, 0.5, 0.0],
                       [3.0, -1.0, 0.0, 1.0],
                       [0.0, 0.0, 0.0, 0.0]])
targets = torch.tensor([1, 0, 3])
my_ce = None  # TODO

In [ ]:
assert torch.isclose(my_ce, nn.CrossEntropyLoss()(logits, targets)), my_ce
print("Rätt!", my_ce.item())

**Fråga:** det tredje exemplet har bara nollor som logits. Vilken loss får det, och varför blir den exakt $\ln 4$?

### Övning 2 – slå baslinjen med Adam och mini-batcher
Träna `nn.Linear(8, 1)` med **Adam**, mini-batcher om 64 och valfritt antal epoker, så att val RMSE blir högst 1 % sämre än sklearns `LinearRegression`.

In [ ]:
set_seed(42)
model = nn.Linear(8, 1).to(device)
# TODO: optimizer, träningsloop med mini-batcher
val_rmse = None  # TODO: beräkna på Xva, yva

In [ ]:
assert val_rmse < rmse_lin * 1.01, f"{val_rmse:.4f} jämfört med {rmse_lin:.4f}"
print(f"Rätt! val RMSE {val_rmse:.4f}")

### Övning 3 – logistisk regression i PyTorch
Klassificera breast cancer-datan (binär) med `nn.Linear(30, 1)` och `nn.BCEWithLogitsLoss`.
- Målet `y` ska vara `float32` med shape `(N, 1)`.
- Prediktera klass 1 om logit > 0, vilket motsvarar sannolikhet > 0.5.

In [ ]:
from sklearn.datasets import load_breast_cancer
Xb, yb = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_va, yb_tr, yb_va = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=42)

# TODO: standardisera (fit på train), gör tensorer, träna, beräkna val accuracy
val_acc = None

In [ ]:
assert val_acc > 0.95, val_acc
print(f"Rätt! val accuracy {val_acc:.3f}")

### Övning 4 – reflektion
1. Varför finns det ingen softmax i sista lagret när vi använder `nn.CrossEntropyLoss`?
2. Din loss blir `nan` efter tre epoker. Vad testar du först?
3. Varför blandar man datan (shuffle) i början av varje epok?

*Dina svar:*

## När använder jag detta?
- **Träningsloopen** (`zero_grad`, forward, loss, `backward`, `step`) används i varje PyTorch-projekt.
- **MSE** för regression, **BCEWithLogits** för binär klassificering och **CrossEntropy** för flera klasser. Alla tar logits eller råa utdata.
- **Adam med `lr=1e-3`** är ett bra första val. Prova SGD med momentum när du vill finjustera.
- **Batchstorlek 32–256** till att börja med. En större batch kräver ofta högre learning rate.
- **Learning rate** är det första du ändrar när träningen inte fungerar.

## Sammanfattning
- Träning är en loop: forward, loss, backward och step, med `zero_grad` varje steg.
- Loss-funktionen väljs efter problemtyp. Klassificering arbetar med logits.
- Learning rate bestämmer steglängden och batchstorleken balansen mellan brus och fart.

**Nästa lektion:** `Dataset` och `DataLoader`, och normalisering av bilddata.